In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/dsn-bootcamp-qualification-hackathon-2026-ml-track/sample_submission.csv
/kaggle/input/competitions/dsn-bootcamp-qualification-hackathon-2026-ml-track/train.csv
/kaggle/input/competitions/dsn-bootcamp-qualification-hackathon-2026-ml-track/test.csv


In [2]:
# DSN Mart Sales Prediction — v9 (structural E30 + CatBoost blend)

## The structural insight we missed

#sales = price × units_sold. Units depend on the store, not the price. One average per store captures ~60% of variance (RMSE ~1073) before any ML at all.

#Three refinements on top:
#- **B**: use mean units (not least-squares slope) — equal weight to all rows
#- **C**: use `product_mean_price` instead of listed price — cancels ~2% listing noise
#- **E30**: add shrunken popularity factor with strength=30 — Bayesian shrinkage toward 1.0, NOT naive target encoding

#**CV verified results (5-fold OOF):**

# | Model | OOF RMSE |
#|---|---|
#| Our old CatBoost baseline | 1075.00 |
#| Our old CatBoost baseline | 1075.00 |
#| CatBoost + product_mean_price | 1074.16 |
#| E30 structural model | 1070.23 |
#| **E30(70%) + CatBoost(30%) blend** | **1069.73** |

#The blend works because the structural model and CatBoost make slightly different errors (correlation 0.9926 vs the useless CatBoost+XGBoost blend at 0.9867).

In [3]:
# note that this is an improvement of the last model
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from catboost import CatBoostRegressor
import warnings
warnings.filterwarnings("ignore")

TARGET = "total_sales"
SEED = 42

In [4]:
train_raw = pd.read_csv("/kaggle/input/competitions/dsn-bootcamp-qualification-hackathon-2026-ml-track/train.csv")
test_raw = pd.read_csv("/kaggle/input/competitions/dsn-bootcamp-qualification-hackathon-2026-ml-track/test.csv")
sample_sub = pd.read_csv("/kaggle/input/competitions/dsn-bootcamp-qualification-hackathon-2026-ml-track/sample_submission.csv")
print("Train:", train_raw.shape, "Test:", test_raw.shape)

Train: (6818, 13) Test: (1705, 12)


In [5]:
# # # 2. Feature engineering

# ##Key fixes over our earlier notebooks:
# ##- `product_category` normalized to lowercase (48 spellings → 16 categories)
# - `product_weight_kg` filled from same product's other rows, then category median
# - `shelf_visibility = 0` treated as missing (526 rows — physically impossible), imputed from product mean then store median
# - `product_mean_price` = average price across all rows of the same product (cancels ~2% listing noise)
# - `store_size` missing filled with "missing" string (CatBoost handles it natively)

# All features computed from train+test combined (features only, target never used here).

In [6]:
def build_features(train_raw, test_raw):
    full = pd.concat([train_raw.drop(columns=[TARGET]), test_raw], ignore_index=True)
    is_train = np.r_[np.ones(len(train_raw), dtype=bool), np.zeros(len(test_raw), dtype=bool)]

    full["category"] = full["product_category"].str.strip().str.lower()
    full["low_fat"] = (full["fat_content"].str.strip().str.lower() == "low fat").astype(int)

    full["weight"] = full["product_weight_kg"].fillna(
        full.groupby("product_code")["product_weight_kg"].transform("mean"))
    full["weight"] = full["weight"].fillna(
        full.groupby("category")["weight"].transform("median"))

    vis = full["shelf_visibility"].mask(full["shelf_visibility"] == 0)
    full["visibility"] = vis.fillna(vis.groupby(full["product_code"]).transform("mean"))
    full["visibility"] = full["visibility"].fillna(
        full.groupby("store_code")["visibility"].transform("median"))

    full["product_mean_price"] = full.groupby("product_code")["product_price"].transform("mean")
    full["price_ratio"] = full["product_price"] / full["product_mean_price"]

    for c in ["category","store_size","store_location_tier","store_format","product_code","store_code"]:
        full[c] = full[c].astype("string").fillna("missing").str.strip().str.lower()

    train = full[is_train].reset_index(drop=True)
    train[TARGET] = train_raw[TARGET].to_numpy()
    test = full[~is_train].reset_index(drop=True)
    return train, test

train, test = build_features(train_raw, test_raw)
print(f"Train {train.shape} | Test {test.shape}")
print(f"Missing values remaining: {train[['weight','visibility','product_mean_price']].isna().sum().to_dict()}")

Train (6818, 19) | Test (1705, 18)
Missing values remaining: {'weight': 0, 'visibility': 0, 'product_mean_price': 0}


In [7]:
# ## 3. Structural model E30

# `prediction = store_avg_units × product_mean_price × popularity_factor(strength=30)`

# The popularity_factor is shrunken toward 1.0 with 30 pseudo-observations — a product with 4 rows averaging 1.20 gets factor = (4×1.20 + 30) / (4 + 30) = 1.024, not 1.20. This is exactly what raw target encoding doesn't do, which is why target encoding failed us catastrophically (CV 929, LB 1215).

In [8]:
def popularity_factor(trn, apply_to, strength, price_col):
    units = trn[TARGET] / trn[price_col]
    rel_units = units / trn["store_code"].map(units.groupby(trn["store_code"]).mean())
    stats = rel_units.groupby(trn["product_code"]).agg(["sum", "count"])
    total = apply_to["product_code"].map(stats["sum"]).fillna(0).to_numpy()
    count = apply_to["product_code"].map(stats["count"]).fillna(0).to_numpy()
    return (total + strength) / (count + strength)

def predict_e30(trn, val):
    store_units = (trn[TARGET] / trn["product_mean_price"]).groupby(trn["store_code"]).mean()
    pred = val["store_code"].map(store_units).to_numpy() * val["product_mean_price"].to_numpy()
    return pred * popularity_factor(trn, val, strength=30, price_col="product_mean_price")

In [9]:
## 4. CatBoost model (with product_mean_price added)

In [10]:
CAT_COLS = ["product_code","category","store_code","store_size","store_location_tier","store_format"]
CB_FEATURES = ["product_code","category","store_code","store_size","store_location_tier",
                "store_format","product_price","product_mean_price","price_ratio",
                "weight","visibility","store_age_years"]
CB_PARAMS = dict(iterations=3000, learning_rate=0.03, depth=6, loss_function="RMSE", random_seed=SEED)

In [11]:
## 5. CV verification — expect ~1069-1070 for the blend

In [12]:
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
y = train[TARGET].to_numpy()

e30_oof = np.zeros(len(train))
cb_oof = np.zeros(len(train))
cb_best_iters = []

for fold, (tr_idx, val_idx) in enumerate(kf.split(train)):
    trn, val = train.iloc[tr_idx], train.iloc[val_idx]

    # E30
    e30_oof[val_idx] = predict_e30(trn, val)

    # CatBoost
    cb = CatBoostRegressor(**CB_PARAMS, cat_features=CAT_COLS, verbose=0)
    cb.fit(trn[CB_FEATURES], trn[TARGET],
           eval_set=(val[CB_FEATURES], val[TARGET]),
           early_stopping_rounds=150, use_best_model=True)
    cb_oof[val_idx] = cb.predict(val[CB_FEATURES])
    cb_best_iters.append(cb.get_best_iteration())

    e30_rmse = np.sqrt(mean_squared_error(val[TARGET], e30_oof[val_idx]))
    cb_rmse  = np.sqrt(mean_squared_error(val[TARGET], cb_oof[val_idx]))
    blend_rmse = np.sqrt(mean_squared_error(val[TARGET], 0.7*e30_oof[val_idx]+0.3*cb_oof[val_idx]))
    print(f"  Fold {fold}: E30={e30_rmse:.2f}  CB={cb_rmse:.2f}  Blend(0.7/0.3)={blend_rmse:.2f}")

mean_best_iter = int(np.mean(cb_best_iters))
blend_oof = 0.7*e30_oof + 0.3*cb_oof
print(f"\n  E30 OOF RMSE:   {np.sqrt(mean_squared_error(y, e30_oof)):.2f}")
print(f"  CB OOF RMSE:    {np.sqrt(mean_squared_error(y, cb_oof)):.2f}")
print(f"  Blend OOF RMSE: {np.sqrt(mean_squared_error(y, blend_oof)):.2f}  (target: ~1069)")
print(f"  CB mean best_iter: {mean_best_iter}")

  Fold 0: E30=1059.95  CB=1064.62  Blend(0.7/0.3)=1059.72
  Fold 1: E30=1045.40  CB=1050.96  Blend(0.7/0.3)=1045.31
  Fold 2: E30=1085.56  CB=1085.58  Blend(0.7/0.3)=1084.08
  Fold 3: E30=1103.79  CB=1104.73  Blend(0.7/0.3)=1102.30
  Fold 4: E30=1055.39  CB=1064.08  Blend(0.7/0.3)=1056.27

  E30 OOF RMSE:   1070.23
  CB OOF RMSE:    1074.16
  Blend OOF RMSE: 1069.73  (target: ~1069)
  CB mean best_iter: 216


In [13]:
# E30: trained on full data
e30_test_pred = predict_e30(train, test)

# CatBoost: 5-seed average for variance reduction
SEEDS = [42, 1, 7, 123, 2024]
cb_test_preds = []
for seed in SEEDS:
    final_params = {**CB_PARAMS, "iterations": mean_best_iter, "random_seed": seed}
    cb = CatBoostRegressor(**final_params, cat_features=CAT_COLS, verbose=0)
    cb.fit(train[CB_FEATURES], train[TARGET])
    cb_test_preds.append(cb.predict(test[CB_FEATURES]))
    print(f"Trained CatBoost seed {seed}")

cb_test_pred = np.mean(cb_test_preds, axis=0)

# Final blend 70% E30 + 30% CatBoost
final_pred = 0.7 * e30_test_pred + 0.3 * cb_test_pred
print(f"\nE30 mean: {e30_test_pred.mean():.1f} | CB mean: {cb_test_pred.mean():.1f} | Blend mean: {final_pred.mean():.1f}")

Trained CatBoost seed 42
Trained CatBoost seed 1
Trained CatBoost seed 7
Trained CatBoost seed 123
Trained CatBoost seed 2024

E30 mean: 2191.2 | CB mean: 2196.5 | Blend mean: 2192.8


In [14]:
submission = pd.DataFrame({"id": test["id"], "total_sales": final_pred})
assert submission.shape[0] == sample_sub.shape[0]
assert list(submission.columns) == list(sample_sub.columns)

submission.to_csv("submission.csv", index=False)
print(f"Saved submission.csv — {submission.shape[0]} rows")
print(f"Predictions: mean={final_pred.mean():.1f}, std={final_pred.std():.1f}, min={final_pred.min():.1f}, max={final_pred.max():.1f}")
submission.head()

Saved submission.csv — 1705 rows
Predictions: mean=2192.8, std=1320.0, min=70.2, max=6802.8


,id,total_sales
0,row_00009,2770.486129
1,row_00015,4519.219812
2,row_00019,2988.516670
3,row_00020,3157.585942
4,row_00023,2475.146238


In [15]:
# ## My Approach — DSN Mart Sales Prediction

# ### Where I started

# My first instinct was to throw everything at a gradient boosting model (CatBoost) and let it figure things out. That got me to roughly 1077 RMSE on the leaderboard. Not bad as a starting point, but I knew I was just pattern-matching without really understanding the data.

# The turning point was stepping back and asking: *what is total_sales actually made of?*

# ---

# ### The key insight: sales = price × units sold

# When I plotted sales against price for each store format, the shape wasn't random — it was a fan. The spread grew with price, and each store had its own fan. That's the signature of multiplication, not addition.

# So I divided total_sales by product_price for every row. What I got back was roughly the number of units sold. And when I looked at that number by store, something clicked:

# | Store | Format | Avg units/product |
# |---|---|---|
# | STORE-JOR / STORE-T5G | Corner Shop | ~2.4 |
# | STORE-HL7 | Superstore | ~13.9 |
# | Standard Supermarkets (6 stores) | Standard | ~15.5–17.1 |
# | STORE-7WS | Flagship Hypermarket | ~26.4 |

# Each store has a characteristic "volume" — how many units of a typical product it moves. Price doesn't change that volume much. The corner shops sell about 2 units per product. The flagship sells about 26. One number per store explains most of the variance.

# ---

# ### Feature engineering decisions

# **What I kept:**
# - `product_mean_price` instead of the listed price — the same product's listed price varies by about 2% across stores (noise from data collection). Averaging across all rows of that product gives a cleaner price signal.
# - Shrunken product popularity factor — some products genuinely sell more than their store's average, but with only ~4 rows per product in training data, a raw average is too noisy. I applied Bayesian shrinkage toward 1.0 with strength=30: a product with 4 rows averaging 1.20 relative units gets a factor of (4×1.20 + 30) / (4 + 30) = 1.024, not 1.20. This reflects the actual uncertainty.

# **What I removed or corrected:**
# - `shelf_visibility = 0` — 526 rows had zero visibility, which is physically impossible for a product being sold. These were treated as missing and imputed from the product's mean visibility across other stores.
# - `store_size` (missing for 3 entire stores) — since `store_code` already uniquely identifies every store attribute, `store_size` adds nothing that `store_code` doesn't already carry.
# - Category, fat content, weight, price bands — after removing price and store effects, the correlation of every remaining feature with units sold collapsed to near zero. Including them adds noise without signal.

# ---

# ### What I tried that didn't work

# I spent time testing target encoding for `product_code`, which looked amazing in cross-validation (RMSE dropped ~145 points) but caused the leaderboard score to get worse — not better. The reason: with only 4–5 rows per product, the "mean" is mostly noise, and keeping the raw category alongside its own encoding let the model reconstruct which fold each row came from. Classic leakage that CV can't catch.

# I also tried log-transforming the target (over-corrects the skew), Tweedie loss, format-specific models, hyperparameter tuning with Optuna across 40 trials, and a CatBoost + XGBoost blend. All neutral or worse. The data was telling me that the problem wasn't the model — it was the problem framing.

# ---

# ### Final model

# **Structural model (E30):** `prediction = store_avg_units × product_mean_price × popularity_factor`

# This is interpretable. You can read off the 10 store multipliers directly. Most products get a popularity factor between 0.98 and 1.02 — the shrinkage keeps the model honest about what it actually knows.

# **Blended with CatBoost (70/30):** The structural model and CatBoost make slightly different errors on some rows (error correlation 0.99 rather than 1.0), so a weighted blend gives a small consistent improvement.

# **Cross-validated RMSE: ~1069–1070**, compared to ~1075 with plain CatBoost. The improvement came from understanding what the data is, not from tuning.
